In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE payment_gateway_catalog.gold.gold_gateway_daily_performance AS
    SELECT
      event_date AS transaction_date,
      provider,
      COUNT(transaction_id) AS total_transactions,
      SUM(CASE WHEN is_success THEN 1 ELSE 0 END) AS successful_transactions,
      ROUND(SUM(CASE WHEN is_success THEN 1 ELSE 0 END) * 100.0 / COUNT(transaction_id), 2) AS success_rate_pct,
      ROUND(SUM(amount_inr), 2) AS gross_volume_inr,
      ROUND(SUM(CASE WHEN is_success THEN amount_inr ELSE 0 END), 2) AS settled_volume_inr
    FROM payment_gateway_catalog.silver.silver_transactions
    GROUP BY event_date, provider
""")

spark.sql("""
    CREATE OR REPLACE TABLE payment_gateway_catalog.gold.gold_failure_root_causes AS
    SELECT
        event_date AS transaction_date,
        provider,
        COALESCE(error_code, 'UNKONWN_ERROR') AS error_code,
        COALESCE(error_description, 'No detailed description provided.') AS error_description,
        COUNT(transaction_id) AS occurence_count
FROM payment_gateway_catalog.silver.silver_transactions
WHERE is_success = FALSE
GROUP BY event_date, provider, error_code, error_description
""")

print("Gold aggregated data marts successfully generated.")

In [0]:
# Display summary metrics
display(spark.sql("SELECT * FROM payment_gateway_catalog.gold.gold_gateway_daily_performance ORDER BY transaction_date DESC, provider"))
display(spark.sql("SELECT * FROM payment_gateway_catalog.gold.gold_failure_root_causes ORDER BY occurence_count DESC"))